**© Copyright AIDENTIFY. All rights reserved.**

본 자료는 **멀티캠퍼스 LLM 파인튜닝 과정** 수강생을 위해 제작되었으며, 강의 목적으로만 사용 가능합니다.  
무단 복제, 배포, 상업적 이용을 금지합니다.

---

# Session 19: LoRA vs Full Fine-tuning 이론 비교

## 🎯 LoRA vs Full Fine-tuning 비교

이번 세션에서는 **LoRA**와 **Full Fine-tuning(FFT)**을 이론적, 실험적으로 비교합니다.

### 두 방법의 핵심 차이

| 구분 | LoRA | Full Fine-tuning |
|------|------|------------------|
| 학습 파라미터 | 전체의 ~1% | 전체 100% |
| 메모리 사용 | 낮음 (4bit + LoRA) | 높음 (FP16/FP32) |
| 학습 속도 | 빠름 | 느림 |
| 저장 크기 | ~수십 MB (어댑터만) | ~수 GB (전체 모델) |
| 성능 | 약간 낮을 수 있음 | 최고 성능 가능 |
| RTX 4060 가능 모델 | ~7B (QLoRA) | ~1.5B |

### 학습 목표

- ✅ LoRA와 FFT의 파라미터 수 차이를 직접 확인
- ✅ 메모리 사용량 비교 (GPU 모니터링)
- ✅ 학습 속도 비교
- ✅ 저장 크기 비교
- ✅ 상황별 선택 가이드 이해

## 1️⃣ 파라미터 수 비교 (전체 vs LoRA 학습 파라미터)

먼저 Qwen2.5-1.5B 모델의 전체 파라미터 수와 LoRA가 학습하는 파라미터 수를 비교합니다.

In [1]:
# 필수 라이브러리
import torch
import gc
import os
import sys
import time
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset
import warnings
warnings.filterwarnings('ignore')

print("✅ 라이브러리 임포트 완료")

/home/ejkim/LLM_Advanced/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/ejkim/LLM_Advanced/venv/lib/python3.12/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


✅ 라이브러리 임포트 완료


In [2]:
# GPU 메모리 모니터링 함수
def print_gpu_memory(tag=""):
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"[{tag}] GPU: {allocated:.1f}GB / {total:.1f}GB")

def get_gpu_memory_gb():
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / 1024**3
    return 0

print_gpu_memory("시작")

[시작] GPU: 0.0GB / 23.5GB


In [3]:
# Qwen2.5-1.5B 파라미터 분석
MODEL_NAME = "Qwen/Qwen2.5-1.5B"

print("📊 Qwen2.5-1.5B 모델 파라미터 분석")
print("="*60)

# 1.5B 모델 구조 정보 (문서 기반)
model_info = {
    "총 파라미터": 1_500_000_000,
    "hidden_size": 1536,
    "num_layers": 28,
    "num_attention_heads": 12,
    "intermediate_size": 8960,
}

# LoRA 파라미터 계산
r = 16
hidden_size = model_info["hidden_size"]
num_layers = model_info["num_layers"]

# 어텐션 레이어: q, k, v, o_proj (4개)
attn_lora_params = 2 * r * hidden_size * 4 * num_layers

# FFN 레이어: gate, up, down_proj (3개) - intermediate_size 사용
intermediate_size = model_info["intermediate_size"]
ffn_lora_gate_up = 2 * r * intermediate_size * 2 * num_layers  # gate, up
ffn_lora_down = 2 * r * hidden_size * 1 * num_layers  # down

# 전체 LoRA 파라미터
attn_only_lora = attn_lora_params
full_lora = attn_lora_params + ffn_lora_gate_up + ffn_lora_down

total_params = model_info["총 파라미터"]

print(f"\n🔹 전체 모델 파라미터: {total_params:,} ({total_params/1e9:.1f}B)")
print(f"\n🔹 LoRA (어텐션만, r={r}):")
print(f"   학습 파라미터: {attn_only_lora:,} ({attn_only_lora/total_params*100:.3f}%)")
print(f"\n🔹 LoRA (어텐션+FFN, r={r}):")
print(f"   학습 파라미터: {full_lora:,} ({full_lora/total_params*100:.3f}%)")
print(f"\n🔹 Full Fine-tuning:")
print(f"   학습 파라미터: {total_params:,} (100.000%)")
print(f"\n📌 LoRA는 전체의 약 {full_lora/total_params*100:.2f}%만 학습! ({total_params//full_lora}배 적음)")

📊 Qwen2.5-1.5B 모델 파라미터 분석

🔹 전체 모델 파라미터: 1,500,000,000 (1.5B)

🔹 LoRA (어텐션만, r=16):
   학습 파라미터: 5,505,024 (0.367%)

🔹 LoRA (어텐션+FFN, r=16):
   학습 파라미터: 22,937,600 (1.529%)

🔹 Full Fine-tuning:
   학습 파라미터: 1,500,000,000 (100.000%)

📌 LoRA는 전체의 약 1.53%만 학습! (65배 적음)


## 2️⃣ LoRA 실습: 모델 로드 → LoRA 적용 → 파라미터 확인

In [4]:
# 공통 데이터 준비 (간단한 텍스트 데이터)
sample_texts = [
    "인공지능은 인간의 지능을 모방하여 학습, 추론, 판단 등의 작업을 수행하는 시스템입니다.",
    "머신러닝은 데이터로부터 패턴을 학습하는 알고리즘을 연구하는 분야입니다.",
    "딥러닝은 인공 신경망을 여러 층으로 쌓아 복잡한 패턴을 학습합니다.",
    "트랜스포머는 셀프 어텐션 메커니즘을 핵심으로 하는 아키텍처입니다.",
    "LoRA는 적은 파라미터만 학습하면서도 좋은 성능을 달성할 수 있습니다.",
] * 10  # 50개로 복제

print(f"✅ 샘플 데이터 준비 완료: {len(sample_texts)}개")

✅ 샘플 데이터 준비 완료: 50개


In [5]:
# LoRA 모델 로드 (4bit 양자화)
print("="*60)
print("📊 LoRA 모델 로드")
print("="*60)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

torch.cuda.empty_cache()
gc.collect()
mem_before_lora = get_gpu_memory_gb()

lora_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
lora_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
lora_model.enable_input_require_grads()

mem_after_load = get_gpu_memory_gb()
print(f"📌 4bit 모델 로드 GPU 메모리: {mem_after_load:.1f}GB")

📊 LoRA 모델 로드
📌 4bit 모델 로드 GPU 메모리: 1.1GB


In [6]:
# LoRA 어댑터 적용
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

lora_model = get_peft_model(lora_model, lora_config)
lora_model.print_trainable_parameters()

mem_after_lora = get_gpu_memory_gb()
print(f"\n📌 LoRA 적용 후 GPU 메모리: {mem_after_lora:.1f}GB")
print(f"📌 LoRA 추가 메모리: {mem_after_lora - mem_after_load:.2f}GB")

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820

📌 LoRA 적용 후 GPU 메모리: 1.1GB
📌 LoRA 추가 메모리: 0.07GB


In [7]:
# LoRA 모델의 레이어 구조 확인
print("📊 LoRA 모델 레이어 구조 (첫 번째 Transformer 블록)")
print("="*60)
for name, param in lora_model.named_parameters():
    if "layers.0." in name:
        status = "🟢 학습" if param.requires_grad else "🔴 동결"
        print(f"  {status} {name}: {param.shape}, {param.dtype}")

📊 LoRA 모델 레이어 구조 (첫 번째 Transformer 블록)
  🔴 동결 base_model.model.model.layers.0.self_attn.q_proj.base_layer.weight: torch.Size([1179648, 1]), torch.uint8
  🔴 동결 base_model.model.model.layers.0.self_attn.q_proj.base_layer.bias: torch.Size([1536]), torch.float16
  🟢 학습 base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight: torch.Size([16, 1536]), torch.float32
  🟢 학습 base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight: torch.Size([1536, 16]), torch.float32
  🔴 동결 base_model.model.model.layers.0.self_attn.k_proj.base_layer.weight: torch.Size([196608, 1]), torch.uint8
  🔴 동결 base_model.model.model.layers.0.self_attn.k_proj.base_layer.bias: torch.Size([256]), torch.float16
  🟢 학습 base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight: torch.Size([16, 1536]), torch.float32
  🟢 학습 base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight: torch.Size([256, 16]), torch.float32
  🔴 동결 base_model.model.model.layers.0.self_attn.v_proj.

## 3️⃣ FFT 실습: 모델 로드 → 전체 파라미터 학습 설정

⚠️ RTX 4060에서 FFT는 1.5B까지만 가능합니다. FP16으로 로드합니다.

In [8]:
# LoRA 모델 메모리 해제
lora_mem_usage = get_gpu_memory_gb()
del lora_model
gc.collect()
torch.cuda.empty_cache()
print("✅ LoRA 모델 메모리 해제")
print_gpu_memory("LoRA 해제 후")

✅ LoRA 모델 메모리 해제
[LoRA 해제 후] GPU: 0.0GB / 23.5GB


In [9]:
# FFT 모델 로드 (FP16 - 양자화 없음)
print("="*60)
print("📊 FFT 모델 로드 (FP16)")
print("="*60)

mem_before_fft = get_gpu_memory_gb()

fft_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

mem_after_fft = get_gpu_memory_gb()
print(f"📌 FP16 모델 로드 GPU 메모리: {mem_after_fft:.1f}GB")

📊 FFT 모델 로드 (FP16)


`torch_dtype` is deprecated! Use `dtype` instead!


📌 FP16 모델 로드 GPU 메모리: 2.9GB


In [10]:
# FFT: 전체 파라미터 학습 가능으로 설정
trainable_params = 0
all_params = 0
for name, param in fft_model.named_parameters():
    all_params += param.numel()
    param.requires_grad = True  # 전체 파라미터 학습
    trainable_params += param.numel()

print(f"📊 FFT 파라미터 정보")
print(f"📌 전체 파라미터: {all_params:,}")
print(f"📌 학습 파라미터: {trainable_params:,}")
print(f"📌 학습 비율: {trainable_params/all_params*100:.1f}%")
print_gpu_memory("FFT 설정 후")

📊 FFT 파라미터 정보
📌 전체 파라미터: 1,543,714,304
📌 학습 파라미터: 1,543,714,304
📌 학습 비율: 100.0%
[FFT 설정 후] GPU: 2.9GB / 23.5GB


In [11]:
# FFT 모델 레이어 구조 확인
print("📊 FFT 모델 레이어 구조 (첫 번째 Transformer 블록)")
print("="*60)
for name, param in fft_model.named_parameters():
    if "layers.0." in name:
        status = "🟢 학습" if param.requires_grad else "🔴 동결"
        print(f"  {status} {name}: {param.shape}, {param.dtype}")

📊 FFT 모델 레이어 구조 (첫 번째 Transformer 블록)
  🟢 학습 model.layers.0.self_attn.q_proj.weight: torch.Size([1536, 1536]), torch.float16
  🟢 학습 model.layers.0.self_attn.q_proj.bias: torch.Size([1536]), torch.float16
  🟢 학습 model.layers.0.self_attn.k_proj.weight: torch.Size([256, 1536]), torch.float16
  🟢 학습 model.layers.0.self_attn.k_proj.bias: torch.Size([256]), torch.float16
  🟢 학습 model.layers.0.self_attn.v_proj.weight: torch.Size([256, 1536]), torch.float16
  🟢 학습 model.layers.0.self_attn.v_proj.bias: torch.Size([256]), torch.float16
  🟢 학습 model.layers.0.self_attn.o_proj.weight: torch.Size([1536, 1536]), torch.float16
  🟢 학습 model.layers.0.mlp.gate_proj.weight: torch.Size([8960, 1536]), torch.float16
  🟢 학습 model.layers.0.mlp.up_proj.weight: torch.Size([8960, 1536]), torch.float16
  🟢 학습 model.layers.0.mlp.down_proj.weight: torch.Size([1536, 8960]), torch.float16
  🟢 학습 model.layers.0.input_layernorm.weight: torch.Size([1536]), torch.float16
  🟢 학습 model.layers.0.post_attention_layernorm.weig

## 4️⃣ 메모리 사용량 비교 (GPU 모니터링)

In [12]:
# FFT 메모리 해제
fft_mem_usage = get_gpu_memory_gb()
del fft_model
gc.collect()
torch.cuda.empty_cache()

# 메모리 비교 결과
print("="*60)
print("📊 GPU 메모리 사용량 비교")
print("="*60)
print(f"\n🔹 LoRA (4bit + LoRA):  ~{lora_mem_usage:.1f}GB")
print(f"🔹 FFT (FP16):          ~{fft_mem_usage:.1f}GB")
print(f"\n📌 FFT는 LoRA 대비 약 {fft_mem_usage/max(lora_mem_usage, 0.1):.1f}배 메모리 사용")
print(f"📌 RTX 4060 (8GB)에서:")
print(f"   - LoRA: ✅ 여유 있음 (학습 시 ~{lora_mem_usage+1:.1f}GB 예상)")
print(f"   - FFT:  ⚠️ 빠듯함 (학습 시 ~{fft_mem_usage+2:.1f}GB 예상)")

📊 GPU 메모리 사용량 비교

🔹 LoRA (4bit + LoRA):  ~1.1GB
🔹 FFT (FP16):          ~2.9GB

📌 FFT는 LoRA 대비 약 2.5배 메모리 사용
📌 RTX 4060 (8GB)에서:
   - LoRA: ✅ 여유 있음 (학습 시 ~2.1GB 예상)
   - FFT:  ⚠️ 빠듯함 (학습 시 ~4.9GB 예상)


## 5️⃣ 학습 속도 비교 (동일 데이터, 동일 에포크)

동일한 데이터와 설정으로 LoRA와 FFT의 학습 속도를 비교합니다.

In [13]:
# 공통 데이터셋 준비
def prepare_dataset(tokenizer, texts, max_length=512):
    """텍스트를 토큰화하여 데이터셋 생성"""
    tokenized = tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=max_length,
        return_tensors=None
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return Dataset.from_dict(tokenized)

dataset = prepare_dataset(tokenizer, sample_texts, max_length=256)
print(f"✅ 데이터셋 준비 완료: {len(dataset)}개 샘플")

✅ 데이터셋 준비 완료: 50개 샘플


In [14]:
# LoRA 학습 시간 측정
print("="*60)
print("⏱️ LoRA 학습 시간 측정")
print("="*60)

# LoRA 모델 재로드
lora_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
lora_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
lora_model.enable_input_require_grads()
lora_model = get_peft_model(lora_model, lora_config)

lora_training_args = TrainingArguments(
    output_dir="./output/lora_speed_test",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    fp16=False,
    logging_steps=5,
    report_to="none",
    remove_unused_columns=False,
    gradient_checkpointing=True,
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
lora_trainer = Trainer(
    model=lora_model,
    args=lora_training_args,
    train_dataset=dataset,
    data_collator=data_collator
)

print("🚀 LoRA 학습 시작...")
lora_start = time.time()
lora_result = lora_trainer.train()
lora_time = time.time() - lora_start
lora_loss = lora_result.training_loss

print(f"\n✅ LoRA 학습 완료")
print(f"📌 소요 시간: {lora_time:.1f}초")
print(f"📌 Training Loss: {lora_loss:.4f}")
print_gpu_memory("LoRA 학습 후")

# 메모리 해제
lora_peak_mem = get_gpu_memory_gb()
del lora_model, lora_trainer
gc.collect()
torch.cuda.empty_cache()

⏱️ LoRA 학습 시간 측정


/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


🚀 LoRA 학습 시작...


Step,Training Loss
5,1.966900



✅ LoRA 학습 완료
📌 소요 시간: 23.8초
📌 Training Loss: 1.9232
[LoRA 학습 후] GPU: 1.3GB / 23.5GB


In [ ]:
# FFT 학습 시간 측정
print("="*60)
print("⏱️ FFT 학습 시간 측정")
print("="*60)

# FFT 모델 로드 (FP16)
fft_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
fft_model.gradient_checkpointing_enable()

fft_training_args = TrainingArguments(
    output_dir="./output/fft_speed_test",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    fp16=False,
    optim="adamw_bnb_8bit",
    logging_steps=5,
    report_to="none",
    remove_unused_columns=False,
    gradient_checkpointing=True,
)

fft_trainer = Trainer(
    model=fft_model,
    args=fft_training_args,
    train_dataset=dataset,
    data_collator=data_collator
)

print("🚀 FFT 학습 시작...")
fft_start = time.time()
fft_result = fft_trainer.train()
fft_time = time.time() - fft_start
fft_loss = fft_result.training_loss

print(f"\n✅ FFT 학습 완료")
print(f"📌 소요 시간: {fft_time:.1f}초")
print(f"📌 Training Loss: {fft_loss:.4f}")
print_gpu_memory("FFT 학습 후")

fft_peak_mem = get_gpu_memory_gb()

⏱️ FFT 학습 시간 측정


The model is already on multiple devices. Skipping the move to device specified in `args`.


🚀 FFT 학습 시작...


Step,Training Loss
5,0.941000
10,0.011400



✅ FFT 학습 완료
📌 소요 시간: 46.9초
📌 Training Loss: 0.3403
[FFT 학습 후] GPU: 7.2GB / 23.5GB


## 6️⃣ 저장 크기 비교

In [16]:
# 모델 저장 및 크기 비교
import shutil

# FFT 모델 저장
fft_save_path = "./output/fft_speed_test/saved_model"
print("⏳ FFT 모델 저장 중...")
fft_model.save_pretrained(fft_save_path)

# FFT 저장 크기 계산
fft_size = 0
for root, dirs, files in os.walk(fft_save_path):
    for f in files:
        fft_size += os.path.getsize(os.path.join(root, f))

# 메모리 해제
del fft_model, fft_trainer
gc.collect()
torch.cuda.empty_cache()

# LoRA 모델 저장 (재로드 필요)
print("⏳ LoRA 어댑터 저장을 위해 모델 재로드...")
lora_model_2 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
lora_model_2 = get_peft_model(lora_model_2, lora_config)

lora_save_path = "./output/lora_speed_test/lora_adapter"
lora_model_2.save_pretrained(lora_save_path)

# LoRA 저장 크기 계산
lora_size = 0
for root, dirs, files in os.walk(lora_save_path):
    for f in files:
        lora_size += os.path.getsize(os.path.join(root, f))

# 크기 비교
print(f"\n📊 저장 크기 비교")
print(f"="*60)
print(f"🔹 LoRA 어댑터:  {lora_size/1024/1024:.1f} MB")
print(f"🔹 FFT 전체 모델: {fft_size/1024/1024:.1f} MB ({fft_size/1024/1024/1024:.2f} GB)")
print(f"\n📌 FFT는 LoRA 대비 약 {fft_size/max(lora_size, 1):.0f}배 큰 저장 공간 필요")

# 정리
del lora_model_2
gc.collect()
torch.cuda.empty_cache()

# 임시 파일 정리
for path in ["./output/lora_speed_test", "./output/fft_speed_test"]:
    if os.path.exists(path):
        shutil.rmtree(path)
print("✅ 임시 파일 정리 완료")

⏳ FFT 모델 저장 중...
⏳ LoRA 어댑터 저장을 위해 모델 재로드...

📊 저장 크기 비교
🔹 LoRA 어댑터:  70.5 MB
🔹 FFT 전체 모델: 2944.4 MB (2.88 GB)

📌 FFT는 LoRA 대비 약 42배 큰 저장 공간 필요
✅ 임시 파일 정리 완료


## 7️⃣ 결과 분석 및 선택 가이드

In [17]:
# 종합 비교표
print("="*70)
print("📊 LoRA vs FFT 종합 비교 결과")
print("="*70)
print(f"\n{'항목':<20} {'LoRA':<25} {'FFT':<25}")
print("-"*70)
print(f"{'학습 파라미터':<20} {'~1% (수십만 개)':<25} {'100% (15억 개)':<25}")
print(f"{'GPU 메모리':<20} {f'~{lora_peak_mem:.1f}GB':<25} {f'~{fft_peak_mem:.1f}GB':<25}")
print(f"{'학습 시간 (1 epoch)':<20} {f'{lora_time:.1f}초':<25} {f'{fft_time:.1f}초':<25}")
print(f"{'Training Loss':<20} {f'{lora_loss:.4f}':<25} {f'{fft_loss:.4f}':<25}")
print(f"{'저장 크기':<20} {f'{lora_size/1024/1024:.1f}MB':<25} {f'{fft_size/1024/1024:.0f}MB':<25}")
print(f"{'RTX 4060 최대 모델':<20} {'7B (QLoRA)':<25} {'1.5B':<25}")
print("-"*70)

📊 LoRA vs FFT 종합 비교 결과

항목                   LoRA                      FFT                      
----------------------------------------------------------------------
학습 파라미터              ~1% (수십만 개)               100% (15억 개)             
GPU 메모리              ~1.3GB                    ~7.2GB                   
학습 시간 (1 epoch)      23.8초                     46.9초                    
Training Loss        1.9232                    0.3403                   
저장 크기                70.5MB                    2944MB                   
RTX 4060 최대 모델       7B (QLoRA)                1.5B                     
----------------------------------------------------------------------


In [18]:
# 선택 가이드
print("\n📋 LoRA vs FFT 선택 가이드")
print("="*60)

print("""
🔹 LoRA를 선택해야 하는 경우:
   ✅ GPU 메모리가 제한적 (8GB 이하)
   ✅ 큰 모델을 사용해야 하는 경우 (7B+)
   ✅ 여러 태스크용 어댑터를 별도 관리하고 싶은 경우
   ✅ 빠른 실험/프로토타이핑이 필요한 경우
   ✅ 원본 모델을 보존하면서 커스터마이징하고 싶은 경우

🔹 FFT를 선택해야 하는 경우:
   ✅ 충분한 GPU 메모리가 있는 경우 (24GB+)
   ✅ 최고 성능이 필요한 경우
   ✅ 모델 구조를 크게 변경해야 하는 경우
   ✅ 데이터가 충분히 많은 경우
   ✅ 특정 도메인에 완전히 특화시켜야 하는 경우

📌 실무 권장사항:
   → 먼저 LoRA로 빠르게 실험하고,
   → 성능이 부족하면 FFT를 고려하세요!
""")


📋 LoRA vs FFT 선택 가이드

🔹 LoRA를 선택해야 하는 경우:
   ✅ GPU 메모리가 제한적 (8GB 이하)
   ✅ 큰 모델을 사용해야 하는 경우 (7B+)
   ✅ 여러 태스크용 어댑터를 별도 관리하고 싶은 경우
   ✅ 빠른 실험/프로토타이핑이 필요한 경우
   ✅ 원본 모델을 보존하면서 커스터마이징하고 싶은 경우

🔹 FFT를 선택해야 하는 경우:
   ✅ 충분한 GPU 메모리가 있는 경우 (24GB+)
   ✅ 최고 성능이 필요한 경우
   ✅ 모델 구조를 크게 변경해야 하는 경우
   ✅ 데이터가 충분히 많은 경우
   ✅ 특정 도메인에 완전히 특화시켜야 하는 경우

📌 실무 권장사항:
   → 먼저 LoRA로 빠르게 실험하고,
   → 성능이 부족하면 FFT를 고려하세요!



## 8️⃣ [보너스] DeepSpeed ZeRO: 멀티 GPU로 FFT 확장하기

> ⚠️ **이 섹션은 듀얼 GPU 환경(TITAN RTX x2) 전용입니다.** 강사 시연용이며, RTX 4060 단일 GPU에서는 실행할 수 없습니다.

### DeepSpeed란?

Microsoft가 개발한 **학습 최적화 프레임워크**로, 핵심은 **ZeRO (Zero Redundancy Optimizer)**입니다.

### ZeRO의 핵심 아이디어

멀티 GPU 학습 시 기존 방식(DDP)은 **모든 GPU에 모델 전체를 복제**합니다. ZeRO는 이를 **분산 저장**합니다:

| 단계 | 분산 대상 | 메모리 절감 |
|------|-----------|------------|
| ZeRO-1 | 옵티마이저 상태 | ~4x |
| ZeRO-2 | + 그래디언트 | ~8x |
| ZeRO-3 | + 모델 파라미터 | GPU 수에 비례 |

```
기존 DDP (GPU 2장):
  GPU 0: [모델 전체] + [옵티마이저 전체] + [그래디언트 전체]
  GPU 1: [모델 전체] + [옵티마이저 전체] + [그래디언트 전체]  ← 중복!

ZeRO-3 (GPU 2장):
  GPU 0: [모델 절반] + [옵티마이저 절반] + [그래디언트 절반]
  GPU 1: [모델 나머지] + [옵티마이저 나머지] + [그래디언트 나머지]  ← 중복 제거!
```

### 왜 중요한가?

앞에서 FFT가 LoRA보다 메모리를 훨씬 많이 사용하는 것을 확인했습니다. **DeepSpeed ZeRO를 쓰면 FFT의 메모리 문제를 멀티 GPU로 해결**할 수 있습니다. 7B 이상의 모델도 FFT가 가능해집니다.

In [19]:
# DeepSpeed 설치 확인 및 GPU 환경 점검
import subprocess

result = subprocess.run(["pip", "show", "deepspeed"], capture_output=True, text=True)
if result.returncode == 0:
    for line in result.stdout.split('\n'):
        if line.startswith(('Name:', 'Version:')):
            print(line)
    print("✅ DeepSpeed 설치 확인 완료")
else:
    print("❌ DeepSpeed가 설치되어 있지 않습니다.")
    print("   설치 명령어: pip install deepspeed")

# GPU 개수 확인
gpu_count = torch.cuda.device_count()
print(f"\n🖥️ 사용 가능한 GPU: {gpu_count}장")
for i in range(gpu_count):
    name = torch.cuda.get_device_name(i)
    mem = torch.cuda.get_device_properties(i).total_memory / 1024**3
    print(f"   GPU {i}: {name} ({mem:.1f}GB)")

if gpu_count < 2:
    print("\n⚠️ GPU가 2장 이상 필요합니다. 이 섹션은 코드만 확인하세요.")
else:
    print(f"\n✅ 멀티 GPU 환경 확인! ZeRO 분산 학습이 가능합니다.")

Name: deepspeed
Version: 0.18.9
✅ DeepSpeed 설치 확인 완료

🖥️ 사용 가능한 GPU: 1장
   GPU 0: NVIDIA TITAN RTX (23.5GB)

⚠️ GPU가 2장 이상 필요합니다. 이 섹션은 코드만 확인하세요.


### DeepSpeed 설정 파일 및 학습 스크립트

DeepSpeed는 **분산 학습 프레임워크**이므로 노트북에서 직접 실행할 수 없고, **별도 Python 스크립트 + `torchrun` 런처**가 필요합니다.

아래 셀에서:
1. ZeRO-2 / ZeRO-3 설정 파일 생성
2. 학습 스크립트 생성
3. `torchrun`으로 2개 GPU에 분산 실행
4. 결과를 읽어서 LoRA / FFT와 비교

In [20]:
import json

# =============================================
# 1) ZeRO-2 설정 파일 생성
# =============================================
zero2_config = {
    "fp16": {"enabled": True},
    "zero_optimization": {
        "stage": 2,
        "allgather_partitions": True,
        "allgather_bucket_size": 2e8,
        "overlap_comm": True,
        "reduce_scatter": True,
        "reduce_bucket_size": 2e8,
        "contiguous_gradients": True
    },
    "gradient_accumulation_steps": 4,
    "gradient_clipping": 1.0,
    "train_micro_batch_size_per_gpu": 1,
    "wall_clock_breakdown": False
}

# =============================================
# 2) ZeRO-3 + CPU Offload 설정 파일 생성
# =============================================
zero3_config = {
    "fp16": {"enabled": True},
    "zero_optimization": {
        "stage": 3,
        "offload_optimizer": {"device": "cpu", "pin_memory": True},
        "offload_param": {"device": "cpu", "pin_memory": True},
        "overlap_comm": True,
        "contiguous_gradients": True,
        "stage3_prefetch_bucket_size": 5e7,
        "stage3_param_persistence_threshold": 1e5,
        "stage3_gather_16bit_weights_on_model_save": True
    },
    "gradient_accumulation_steps": 4,
    "gradient_clipping": 1.0,
    "train_micro_batch_size_per_gpu": 1,
    "wall_clock_breakdown": False
}

# 설정 파일 저장
os.makedirs("./output/deepspeed_demo", exist_ok=True)
for name, config in [("zero2", zero2_config), ("zero3", zero3_config)]:
    path = f"./output/deepspeed_demo/ds_{name}_config.json"
    with open(path, "w") as f:
        json.dump(config, f, indent=2)
    print(f"✅ {path} 생성 완료")

# 설정 비교
print("\n📊 ZeRO-2 vs ZeRO-3 설정 비교")
print("="*60)
print(f"{'항목':<30} {'ZeRO-2':<15} {'ZeRO-3':<15}")
print("-"*60)
print(f"{'옵티마이저 분산':<30} {'✅':<15} {'✅':<15}")
print(f"{'그래디언트 분산':<30} {'✅':<15} {'✅':<15}")
print(f"{'모델 파라미터 분산':<30} {'❌':<15} {'✅':<15}")
print(f"{'CPU Offload':<30} {'❌':<15} {'✅':<15}")
print(f"{'통신 비용':<30} {'낮음':<15} {'높음':<15}")
print(f"{'메모리 절감':<30} {'중간':<15} {'최대':<15}")
print(f"{'속도':<30} {'빠름':<15} {'느림 (offload)':<15}")


✅ ./output/deepspeed_demo/ds_zero2_config.json 생성 완료
✅ ./output/deepspeed_demo/ds_zero3_config.json 생성 완료

📊 ZeRO-2 vs ZeRO-3 설정 비교
항목                             ZeRO-2          ZeRO-3         
------------------------------------------------------------
옵티마이저 분산                       ✅               ✅              
그래디언트 분산                       ✅               ✅              
모델 파라미터 분산                     ❌               ✅              
CPU Offload                    ❌               ✅              
통신 비용                          낮음              높음             
메모리 절감                         중간              최대             
속도                             빠름              느림 (offload)   


In [21]:
# 학습 스크립트 생성 (torchrun으로 실행할 .py 파일)
# DeepSpeed는 멀티프로세스 런처가 필요하므로 노트북이 아닌 스크립트로 실행

train_script = '''
"""DeepSpeed ZeRO FFT 학습 스크립트
Usage: torchrun --nproc_per_node=2 deepspeed_train.py --zero_stage 2
"""
import os
import sys
import json
import time
import argparse
import torch
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, Trainer, DataCollatorForLanguageModeling
)
from datasets import Dataset

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--zero_stage", type=int, default=2, choices=[2, 3])
    parser.add_argument("--model_name", type=str, default="Qwen/Qwen2.5-1.5B")
    parser.add_argument("--local_rank", type=int, default=-1)  # torchrun이 자동 설정
    args = parser.parse_args()

    # 데이터 준비 (앞 섹션과 동일)
    sample_texts = [
        "인공지능은 인간의 지능을 모방하여 학습, 추론, 판단 등의 작업을 수행하는 시스템입니다.",
        "머신러닝은 데이터로부터 패턴을 학습하는 알고리즘을 연구하는 분야입니다.",
        "딥러닝은 인공 신경망을 여러 층으로 쌓아 복잡한 패턴을 학습합니다.",
        "트랜스포머는 셀프 어텐션 메커니즘을 핵심으로 하는 아키텍처입니다.",
        "LoRA는 적은 파라미터만 학습하면서도 좋은 성능을 달성할 수 있습니다.",
    ] * 10

    tokenizer = AutoTokenizer.from_pretrained(args.model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    tokenized = tokenizer(
        sample_texts, truncation=True, padding="max_length",
        max_length=256, return_tensors=None
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    dataset = Dataset.from_dict(tokenized)

    # 모델 로드 (FP16, device_map 없이 - DeepSpeed가 관리)
    model = AutoModelForCausalLM.from_pretrained(
        args.model_name,
        torch_dtype=torch.float16,
        trust_remote_code=True
    )
    model.gradient_checkpointing_enable()

    # DeepSpeed 설정 파일 경로
    ds_config = f"./output/deepspeed_demo/ds_zero{args.zero_stage}_config.json"

    training_args = TrainingArguments(
        output_dir=f"./output/deepspeed_demo/zero{args.zero_stage}_output",
        num_train_epochs=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        fp16=False,
        logging_steps=5,
        save_strategy="no",
        report_to="none",
        remove_unused_columns=False,
        gradient_checkpointing=True,
        deepspeed=ds_config,  # DeepSpeed 활성화!
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset,
        data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
    )

    # 학습 실행 + 시간 측정
    start = time.time()
    result = trainer.train()
    elapsed = time.time() - start

    # rank 0에서만 결과 출력
    if trainer.is_world_process_zero():
        # GPU별 메모리 사용량
        gpu_mem = []
        for i in range(torch.cuda.device_count()):
            mem = torch.cuda.max_memory_allocated(i) / 1024**3
            gpu_mem.append(f"{mem:.1f}GB")

        results = {
            "zero_stage": args.zero_stage,
            "training_loss": result.training_loss,
            "elapsed_seconds": round(elapsed, 1),
            "gpu_memory": gpu_mem,
            "num_gpus": torch.cuda.device_count(),
        }
        result_path = f"./output/deepspeed_demo/zero{args.zero_stage}_result.json"
        with open(result_path, "w") as f:
            json.dump(results, f, indent=2)
        print(f"\n✅ ZeRO-{args.zero_stage} 학습 완료!")
        print(f"📌 소요 시간: {elapsed:.1f}초")
        print(f"📌 Training Loss: {result.training_loss:.4f}")
        print(f"📌 GPU 메모리: {gpu_mem}")
        print(f"📌 결과 저장: {result_path}")

if __name__ == "__main__":
    main()
'''

script_path = "./output/deepspeed_demo/deepspeed_train.py"
with open(script_path, "w") as f:
    f.write(train_script)

print(f"✅ 학습 스크립트 생성: {script_path}")
print(f"   파일 크기: {os.path.getsize(script_path):,} bytes")
print()
print("📋 실행 명령어:")
print("   ZeRO-2: torchrun --nproc_per_node=2 ./output/deepspeed_demo/deepspeed_train.py --zero_stage 2")
print("   ZeRO-3: torchrun --nproc_per_node=2 ./output/deepspeed_demo/deepspeed_train.py --zero_stage 3")


✅ 학습 스크립트 생성: ./output/deepspeed_demo/deepspeed_train.py
   파일 크기: 3,932 bytes

📋 실행 명령어:
   ZeRO-2: torchrun --nproc_per_node=2 ./output/deepspeed_demo/deepspeed_train.py --zero_stage 2
   ZeRO-3: torchrun --nproc_per_node=2 ./output/deepspeed_demo/deepspeed_train.py --zero_stage 3


### ZeRO-2 학습 실행

아래 셀에서 `torchrun`으로 **2개 GPU에 분산 학습**을 실행합니다.

- `--nproc_per_node=2`: GPU 2장 사용
- `--zero_stage 2`: ZeRO Stage 2 (옵티마이저 + 그래디언트 분산)

> 💡 `device_map="auto"`를 사용하지 않습니다. DeepSpeed가 직접 모델을 GPU에 분배하므로, HuggingFace의 `device_map`과 충돌합니다.

In [22]:
# ZeRO-2 학습 실행 (GPU 2장 분산)
import subprocess

if torch.cuda.device_count() >= 2:
    print("🚀 ZeRO-2 분산 학습 시작 (GPU 2장)...")
    print("="*60)

    result = subprocess.run(
        [
            sys.executable, "-m", "torch.distributed.run",
            "--nproc_per_node=2",
            "./output/deepspeed_demo/deepspeed_train.py",
            "--zero_stage", "2"
        ],
        capture_output=True, text=True, timeout=600,
        cwd=os.getcwd()
    )

    # 출력 표시 (rank 0 메시지만 필터링)
    for line in result.stdout.split('\n'):
        if any(k in line for k in ['✅', '📌', 'loss', 'ZeRO', 'Epoch']):
            print(line)

    if result.returncode != 0:
        print("\n❌ 에러 발생:")
        # 마지막 20줄만 출력
        err_lines = result.stderr.strip().split('\n')[-20:]
        for line in err_lines:
            print(f"  {line}")
    else:
        print("\n✅ ZeRO-2 학습 완료!")
else:
    print("⚠️ GPU 2장 이상 필요. 아래는 실행 시 예상 출력입니다:")
    print()
    print("🚀 ZeRO-2 분산 학습 시작 (GPU 2장)...")
    print("✅ ZeRO-2 학습 완료!")
    print("📌 소요 시간: ~21초 (1.5B, 50샘플 기준)")
    print("📌 Training Loss: ~1.15")
    print("📌 GPU 메모리: ['~17.3GB', '~0.0GB']")


⚠️ GPU 2장 이상 필요. 아래는 실행 시 예상 출력입니다:

🚀 ZeRO-2 분산 학습 시작 (GPU 2장)...
✅ ZeRO-2 학습 완료!
📌 소요 시간: ~21초 (1.5B, 50샘플 기준)
📌 Training Loss: ~1.15
📌 GPU 메모리: ['~17.3GB', '~0.0GB']


### ZeRO-3 + CPU Offload 학습 실행

이번에는 **모델 파라미터까지 분산 + CPU로 오프로드**하는 ZeRO-3을 실행합니다.

ZeRO-3은 메모리를 가장 많이 절약하지만, CPU↔GPU 데이터 이동 때문에 **ZeRO-2보다 느립니다**. 이 트레이드오프를 직접 확인해봅시다.

In [23]:
# ZeRO-3 + CPU Offload 학습 실행
import subprocess

if torch.cuda.device_count() >= 2:
    print("🚀 ZeRO-3 + CPU Offload 분산 학습 시작 (GPU 2장)...")
    print("="*60)
    print()
    print("⚠️ ZeRO-3은 CPU offload용 CUDA 커널 컴파일이 필요합니다.")
    print("   현재 환경: 시스템 CUDA 13.2 vs PyTorch CUDA 12.8 → 버전 불일치로 컴파일 실패")
    print("   해결: PyTorch와 동일한 CUDA 버전 설치 또는 DS_BUILD_OPS=0 환경변수 사용")
    print()
    print("📌 ZeRO-3 예상 결과 (정상 환경 기준):")
    print("   소요 시간: ~60초 (CPU offload로 인해 ZeRO-2보다 느림)")
    print("   Training Loss: ~1.15 (ZeRO-2와 동일)")
    print("   GPU 메모리: ~3~5GB/GPU (CPU offload 효과)")
else:
    print("⚠️ GPU 2장 이상 필요. 아래는 실행 시 예상 출력입니다:")
    print()
    print("🚀 ZeRO-3 + CPU Offload 분산 학습 시작 (GPU 2장)...")
    print("📌 소요 시간: ~60초 (CPU offload로 인해 ZeRO-2보다 느림)")
    print("📌 Training Loss: ~1.15")
    print("📌 GPU 메모리: ~3~5GB/GPU (CPU offload 효과)")


⚠️ GPU 2장 이상 필요. 아래는 실행 시 예상 출력입니다:

🚀 ZeRO-3 + CPU Offload 분산 학습 시작 (GPU 2장)...
📌 소요 시간: ~60초 (CPU offload로 인해 ZeRO-2보다 느림)
📌 Training Loss: ~1.15
📌 GPU 메모리: ~3~5GB/GPU (CPU offload 효과)


### 4가지 방식 종합 비교: LoRA vs FFT vs ZeRO-2 vs ZeRO-3

In [24]:
# 4가지 방식 종합 비교
print("="*80)
print("📊 LoRA vs FFT vs FFT+ZeRO-2 vs FFT+ZeRO-3 종합 비교")
print("="*80)

# DeepSpeed 결과 로드
zero2_result, zero3_result = None, None
for stage in [2, 3]:
    path = f"./output/deepspeed_demo/zero{stage}_result.json"
    if os.path.exists(path):
        with open(path) as f:
            if stage == 2:
                zero2_result = json.load(f)
            else:
                zero3_result = json.load(f)

# 비교표 출력
print(f"\n{'항목':<18} {'LoRA':<16} {'FFT':<16} {'FFT+ZeRO-2':<16} {'FFT+ZeRO-3':<16}")
print("-"*82)

# GPU 개수
print(f"{'GPU 수':<18} {'1장':<16} {'1장':<16} {'2장':<16} {'2장':<16}")

# 학습 파라미터
print(f"{'학습 파라미터':<18} {'~1%':<16} {'100%':<16} {'100%':<16} {'100%':<16}")

# 학습 시간
z2_time = f"{zero2_result['elapsed_seconds']}초" if zero2_result else "~30초*"
z3_time = f"{zero3_result['elapsed_seconds']}초" if zero3_result else "~60초*"
try:
    l_time = f"{lora_time:.1f}초"
    f_time = f"{fft_time:.1f}초"
except NameError:
    l_time = "(위 셀 실행 필요)"
    f_time = "(위 셀 실행 필요)"
print(f"{'학습 시간':<18} {l_time:<16} {f_time:<16} {z2_time:<16} {z3_time:<16}")

# Training Loss
z2_loss = f"{zero2_result['training_loss']:.4f}" if zero2_result else "~1.15*"
z3_loss = f"{zero3_result['training_loss']:.4f}" if zero3_result else "~1.15*"
try:
    l_loss = f"{lora_loss:.4f}"
    f_loss = f"{fft_loss:.4f}"
except NameError:
    l_loss = "-"
    f_loss = "-"
print(f"{'Training Loss':<18} {l_loss:<16} {f_loss:<16} {z2_loss:<16} {z3_loss:<16}")

# GPU 메모리
z2_mem = f"{zero2_result['gpu_memory'][0]}/GPU" if zero2_result else "~5.2GB/GPU*"
z3_mem = f"{zero3_result['gpu_memory'][0]}/GPU" if zero3_result else "~3~5GB/GPU*"
try:
    l_mem = f"~{lora_peak_mem:.1f}GB"
    f_mem = f"~{fft_peak_mem:.1f}GB"
except NameError:
    l_mem = "-"
    f_mem = "-"
print(f"{'GPU 메모리':<18} {l_mem:<16} {f_mem:<16} {z2_mem:<16} {z3_mem:<16}")

# 7B 모델 가능 여부
print(f"{'7B FFT 가능?':<18} {'❌ (LoRA만)':<16} {'❌ (OOM)':<16} {'✅ (분산)':<16} {'✅ (분산+오프로드)':<16}")

print("-"*82)
if not zero2_result:
    print("* 표시는 TITAN RTX x2 환경 예상값 (실제 실행 시 갱신됨)")

print("\n📌 핵심 인사이트:")
print("   1️⃣ ZeRO-2: FFT와 동일한 품질 + GPU당 메모리 절반 → 속도도 빠름")
print("   2️⃣ ZeRO-3: 메모리 절감 최대 but CPU offload로 속도 희생")
print("   3️⃣ LoRA: 여전히 가장 빠르고 메모리 효율적 (성능 트레이드오프)")
print("   4️⃣ 실무: LoRA로 시작 → 성능 부족 시 ZeRO-2 FFT → 메모리 부족 시 ZeRO-3")


📊 LoRA vs FFT vs FFT+ZeRO-2 vs FFT+ZeRO-3 종합 비교

항목                 LoRA             FFT              FFT+ZeRO-2       FFT+ZeRO-3      
----------------------------------------------------------------------------------
GPU 수              1장               1장               2장               2장              
학습 파라미터            ~1%              100%             100%             100%            
학습 시간              23.8초            46.9초            ~30초*            ~60초*           
Training Loss      1.9232           0.3403           ~1.15*           ~1.15*          
GPU 메모리            ~1.3GB           ~7.2GB           ~5.2GB/GPU*      ~3~5GB/GPU*     
7B FFT 가능?         ❌ (LoRA만)        ❌ (OOM)          ✅ (분산)           ✅ (분산+오프로드)     
----------------------------------------------------------------------------------
* 표시는 TITAN RTX x2 환경 예상값 (실제 실행 시 갱신됨)

📌 핵심 인사이트:
   1️⃣ ZeRO-2: FFT와 동일한 품질 + GPU당 메모리 절반 → 속도도 빠름
   2️⃣ ZeRO-3: 메모리 절감 최대 but CPU offload로 속도 희생
   3️⃣ LoRA: 여전히 가장 빠르고 메모

In [25]:
# DeepSpeed 임시 파일 정리
import shutil

cleanup_path = "./output/deepspeed_demo"
if os.path.exists(cleanup_path):
    shutil.rmtree(cleanup_path)
    print(f"✅ {cleanup_path} 정리 완료")

# GPU 메모리 최종 정리
gc.collect()
torch.cuda.empty_cache()
print_gpu_memory("DeepSpeed 섹션 종료")


✅ ./output/deepspeed_demo 정리 완료
[DeepSpeed 섹션 종료] GPU: 0.5GB / 23.5GB


## 📝 정리 및 핵심 요약

### 이번 실습에서 배운 내용

| 항목 | LoRA | FFT | FFT + DeepSpeed ZeRO |
|------|------|-----|---------------------|
| 학습 파라미터 | ~1% (수십만 개) | 100% (15억 개) | 100% (15억 개) |
| GPU 수 | 1장 | 1장 | 2장+ (분산) |
| GPU 메모리 | 적음 (4bit + LoRA) | 많음 (FP16 전체) | GPU당 절반 이하 |
| 학습 속도 | 빠름 | 느림 | 중간 (통신 오버헤드) |
| 저장 크기 | ~수십 MB | ~수 GB | ~수 GB |

### 핵심 포인트

- 🎯 **LoRA는 전체 파라미터의 ~1%만 학습**하므로 메모리와 시간이 절약됩니다
- 🎯 **FFT는 최고 성능**을 달성할 수 있지만, 자원이 많이 필요합니다
- 🎯 **DeepSpeed ZeRO는 멀티 GPU로 FFT의 메모리 문제를 해결**합니다
  - ZeRO-2: 속도↑ 메모리 절감 중간 / ZeRO-3: 속도↓ 메모리 절감 최대
- 🎯 **RTX 4060(단일)에서 FFT는 1.5B까지만** 가능하지만, **ZeRO-3 + 멀티GPU면 7B+ FFT 가능**
- 🎯 **실무 권장 순서**: LoRA → ZeRO-2 FFT → ZeRO-3 FFT (메모리 부족 시)

### 다음 단계

- ➡️ **Notebook 20**: LoRA vs FFT 실전 비교 - 동일 데이터로 학습 후 성능 비교
